# JPYC オンチェーン・パネルデータ基盤 (Colab用)

**手順**: ①このノートブックと同じ場所に `config.py / rpc.py / collector.py / panel_builder.py / analyze.py / known_addresses.csv` をアップロード → ②上から順に実行。

- 収集は中断してもOK(チェックポイントから再開)。ただしColabの無料ランタイムは切断されるので、**Googleドライブをマウントして data/ をドライブ上に置く**のが安全(下のセル参照)。
- 所要時間の目安: Avalanche/Kaia は速い。Polygon はブロック数が多く数時間かかることがある。チェーンごとに別セッションで走らせてもよい。

In [ ]:
# (推奨) Googleドライブに作業ディレクトリを置く: ランタイム切断でもデータが残る
from google.colab import drive
drive.mount('/content/drive')
%mkdir -p /content/drive/MyDrive/jpyc_panel
%cd /content/drive/MyDrive/jpyc_panel
# ここに .py ファイル一式をアップロードしておく(左のファイルペインからドラッグ&ドロップ)

In [ ]:
!pip -q install pyarrow

In [ ]:
# ① 収集: まず軽いチェーンで動作確認 → その後、全チェーン
!python collector.py avalanche
!python collector.py kaia
!python collector.py ethereum
!python collector.py polygon   # 一番重い。途中で切れたら同じコマンドで再開できる

In [ ]:
# ② パネル構築(チェーン別 + combined)。「⚠ 負残高」警告が出たら取得漏れなので①をやり直す
!python panel_builder.py

In [ ]:
# ③ 図表生成
!python analyze.py combined

from IPython.display import Image, display
import glob
for f in sorted(glob.glob('output/figs_combined/*.png')):
    display(Image(f, width=850))

In [ ]:
# ④ パネルを直接触る
import pandas as pd
p = pd.read_csv('output/daily_panel_combined.csv', parse_dates=['date'])
p.tail()

## ⑤ 運営・CEXアドレスのラベリング(重要)
`output/flag_candidates_combined.csv` を開き、上位アドレスをエクスプローラー(Etherscan / Polygonscan / Snowtrace / Kaiascan)で確認して、運営・取引所・コントラクトと判明したものを `known_addresses.csv` に `category=exclude` で追記 → `panel_builder.py` を再実行。これで「ユーザーの分布」と「運営込みの分布」を区別できるようになる。